In [1]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate "datasets<3.0.0" zstandard tqdm
!pip install -q lm-eval==0.4.4

import sys
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")
sys.path.insert(0, "/content/llm-quantization-thesis")

from google.colab import drive
drive.mount('/content/drive')

!mkdir -p smoothquant_repo/act_scales
!cp /content/drive/MyDrive/thesis_results/act_scales/opt-125m.pt smoothquant_repo/act_scales/

!mkdir -p act_percentiles/opt-125m
!cp /content/drive/MyDrive/thesis_results/act_percentiles/opt-125m/*.pt act_percentiles/opt-125m/

!nvidia-smi
!ls -la smoothquant_repo/act_scales/ act_percentiles/opt-125m/
!python -c "from smoothquant.smooth import smooth_lm; print('smoothquant OK')"
!python -c "from experiments.task02_percentile_smoothing.percentile_smooth import smooth_lm_pct; print('percentile smooth OK')"
!python -c "import importlib.metadata; print('lm_eval', importlib.metadata.version('lm_eval'))"

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 164 (delta 66), reused 143 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 4.87 MiB | 21.20 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 23.76 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
MODEL = "facebook/opt-125m"
SCRIPT = "experiments/task03_zero_shot_eval/opt_125m/run_zero_shot_t3.py"
MAX_SCALES = "smoothquant_repo/act_scales/opt-125m.pt"

# Alphas (paper default 0.5 for O1/O2)
ALPHA_O1 = 0.5
ALPHA_O2 = 0.5

# Percentile-smoothing knobs — Task 02 OPT-125M winner
P_PCT     = 0.999
ALPHA_PCT = 0.5
PCT_SCALES = f"act_percentiles/opt-125m/p{P_PCT}.pt"

BATCH = 8

OUT_DIR = "results/task03"
!mkdir -p {OUT_DIR}
print(f"Percentile config -> p={P_PCT}, alpha={ALPHA_PCT}, file={PCT_SCALES}")

Percentile config -> p=0.999, alpha=0.5, file=act_percentiles/opt-125m/p0.999.pt


In [3]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --config_label FP16 \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-125m_zeroshot_FP16.json

  Config:        FP16
  Model:         facebook/opt-125m
  Smooth:        False (max, alpha=0.5, p_w=1.0)
  Quant:         False (W=per_channel, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:03:51:36,832 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:03:51:36,838 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
2026-05-08:03:51:36,844 INFO     [_client.py:1025] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
config.json: 100% 651/651 [00:00<00:00, 3.67MB/s]
2026-05-08:03:51:37,097 INFO     [_client.py:1025] HTTP Request: HEAD https://hu

In [4]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label W8A8-naive \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-125m_zeroshot_W8A8-naive.json

  Config:        W8A8-naive
  Model:         facebook/opt-125m
  Smooth:        False (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_tensor, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:03:55:44,569 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:03:55:44,570 WARNING  [_http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08:03:55:44,575 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
2026-05-08:03:55:44,812 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/tokenizer_config.json

In [5]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method max --alpha {ALPHA_O1} \
    --act_scales_path {MAX_SCALES} \
    --quantize \
    --weight_quant per_tensor --act_quant per_token \
    --config_label SQ-O1-max \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-125m_zeroshot_SQ-O1-max.json

  Config:        SQ-O1-max
  Model:         facebook/opt-125m
  Smooth:        True (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:04:00:51,289 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:04:00:51,289 WARNING  [_http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08:04:00:51,301 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
2026-05-08:04:00:51,542 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/tokenizer_config.json "H

In [6]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method max --alpha {ALPHA_O2} \
    --act_scales_path {MAX_SCALES} \
    --quantize \
    --weight_quant per_tensor --act_quant per_tensor \
    --config_label SQ-O2-max \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-125m_zeroshot_SQ-O2-max.json

  Config:        SQ-O2-max
  Model:         facebook/opt-125m
  Smooth:        True (max, alpha=0.5, p_w=1.0)
  Quant:         True (W=per_tensor, A=per_tensor, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:04:05:56,268 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:04:05:56,273 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
2026-05-08:04:05:56,514 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:04:05:56,519 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871

In [7]:
!python {SCRIPT} \
    --model_path {MODEL} \
    --smooth --smooth_method percentile --alpha {ALPHA_PCT} --p_w {P_PCT} \
    --act_scales_path {PCT_SCALES} \
    --quantize \
    --weight_quant per_channel --act_quant per_token \
    --config_label SQ-C-pct \
    --batch_size {BATCH} \
    --save_json {OUT_DIR}/opt-125m_zeroshot_SQ-C-pct.json

  Config:        SQ-C-pct
  Model:         facebook/opt-125m
  Smooth:        True (percentile, alpha=0.5, p_w=0.999)
  Quant:         True (W=per_channel, A=per_token, bmm=True)
  Tasks:         ['lambada_openai', 'hellaswag', 'piqa', 'winogrande', 'openbookqa', 'rte', 'copa']
  Batch size:    8
2026-05-08:04:11:10,522 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08:04:11:10,523 WARNING  [_http.py:904] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08:04:11:10,528 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/opt-125m/27dcfa74d334bc871f3234de431e71c6eeba5dd6/config.json "HTTP/1.1 200 OK"
2026-05-08:04:11:10,766 INFO     [_client.py:1025] HTTP Request: HEAD https://huggingface.co/facebook/opt-125m/resolve/main/tokenizer_confi

In [8]:
!mkdir -p /content/drive/MyDrive/thesis_results/task03
!cp {OUT_DIR}/opt-125m_zeroshot_*.json /content/drive/MyDrive/thesis_results/task03/

import json, glob

TASKS = ["lambada_openai", "hellaswag", "piqa", "winogrande", "openbookqa", "rte", "copa"]
PRIMARY = {
    "lambada_openai": "acc,none",
    "hellaswag":      "acc_norm,none",
    "piqa":           "acc_norm,none",
    "winogrande":     "acc,none",
    "openbookqa":     "acc_norm,none",
    "rte":            "acc,none",
    "copa":           "acc,none",
}
ORDER = ["FP16", "W8A8-naive", "SQ-O1-max", "SQ-O2-max", "SQ-C-pct"]

rows_by_label = {}
for f in sorted(glob.glob(f"{OUT_DIR}/opt-125m_zeroshot_*.json")):
    r = json.load(open(f))
    label = r["config_label"]
    row = {"config": label}
    for t in TASKS:
        m = r["results"].get(t, {})
        v = m.get(PRIMARY[t])
        if v is None:
            for k, val in m.items():
                if isinstance(val, (int, float)):
                    v = val
                    break
        row[t] = v
    nums = [row[t] for t in TASKS if isinstance(row[t], (int, float))]
    row["avg"] = sum(nums) / len(nums) if nums else None
    rows_by_label[label] = row

rows = [rows_by_label[l] for l in ORDER if l in rows_by_label]

header = ["config"] + TASKS + ["avg"]
print("  ".join(f"{h:>14}" for h in header))
print("-" * (16 * len(header)))
for row in rows:
    cells = [f"{row['config']:>14}"]
    for t in TASKS + ["avg"]:
        v = row.get(t)
        cells.append(f"{v:>14.4f}" if isinstance(v, (int, float)) else f"{'-':>14}")
    print("  ".join(cells))

        config  lambada_openai       hellaswag            piqa      winogrande      openbookqa             rte            copa             avg
------------------------------------------------------------------------------------------------------------------------------------------------
          FP16          0.3788          0.3135          0.6192          0.5020          0.2780          0.5018          0.6900          0.4690
    W8A8-naive          0.3538          0.3104          0.6159          0.4972          0.2680          0.4838          0.6300          0.4513
     SQ-O1-max          0.3829          0.3136          0.6181          0.4988          0.2740          0.4765          0.6800          0.4634
     SQ-O2-max          0.3707          0.3121          0.6148          0.5083          0.2600          0.4729          0.6500          0.4555
      SQ-C-pct          0.3749          0.3136          0.6197          0.4941          0.2720          0.4874          0.7000          0.46